# p4m18_pair2_born_again_A8

From-scratch **D4-equivariant (p4m) 18-block** PAIR2 student (+ legal-mask loss, A1 recipe) on the same corpus as the 18x96 born-again run: 12,906,096 states labeled with teacher **A8**'s 8-symmetry-averaged policy (top-5 soft). HISTORY 278. The trunk is equivariant by construction (group convolutions over the 8 rotations/reflections), so one forward pass is already symmetric; the goal is the teacher's 8-view ensemble (0.21 deaths per 100k turns) in a single pass (the 18x96 student: 0.318).

Two width arms, pick one per runtime in the CONFIG cell: `ARM = "g16"` (16 group channels = 128 conv channels, 742k params, 1.64x the 18x96's inference cost) or `ARM = "g12"` (12 = 96 conv channels, 443k params, the 18x96's cost). Both if two runtimes are available.

Drive files (MyDrive/alphatrain/): `colorlines_pillar3d_v9.tar.gz` (new) + `born_again_A8.pt.gz` (already uploaded for the 18x96 run).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v9.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v9.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/born_again_A8.pt.gz /content/born_again_A8.pt.gz
gz=os.path.getsize('/content/born_again_A8.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 873544987, f'.gz truncated! got {gz}; re-upload born_again_A8.pt.gz'
!gunzip -t /content/born_again_A8.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/born_again_A8.pt.gz > /content/alphatrain/data/born_again_A8.pt
pt=os.path.getsize('/content/alphatrain/data/born_again_A8.pt')
assert pt == 2064979169, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 12,906,096 rows ({time.time()-t0:.0f}s)')
!rm /content/born_again_A8.pt.gz
!pip install -q numpy numba scipy
_src = open('/content/alphatrain/dataset.py').read()
assert 'def _build_line_direction_luts' in _src and 'def _transform_observation' in _src, 'STALE CODE TARBALL (old D4 bug)'
_tp = open('/content/alphatrain/train_path_b.py').read()
assert 'class MixedLoader' in _tp and 'def legal_mask_from_obs' in _tp, 'STALE CODE TARBALL (trainer)'
print('code tarball OK')
import os.path as _p
assert _p.exists('/content/alphatrain/model_p4m.py') and '--trunk' in _tp, 'STALE CODE TARBALL (no p4m trunk)'

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG =====
ARM        = "g16"    # "g16": 16 group channels (x8 = 128 conv channels); "g12": 12 (x8 = 96, the 18x96's cost)
GROUP_CH, EXPAND = {"g16": (16, 64), "g12": (12, 48)}[ARM]
NUM_BLOCKS = 18
EPOCHS     = 40
BATCH      = 32768    # OOM on a <80 GB GPU: BATCH = 16384 and LR = 2e-3
LR         = 3e-3
SEED       = 42
SAVE_STEPS = 1000
COMPILE    = "--compile"   # if torch.compile fails on this runtime: COMPILE = ""
RUN        = f"p4m{NUM_BLOCKS}_{ARM}_pair2_born_again_A8"
print(f'RUN={RUN} group_ch={GROUP_CH} expand={EXPAND} ep={EPOCHS} bs={BATCH} lr={LR}')

In [ ]:
# Resume from the last COMPLETED epoch if Colab stopped. Checkpoints go straight to Drive every epoch.
LATEST = f'/content/drive/MyDrive/alphatrain/{RUN}_ckpts/latest.pt'
RESUME_ARGS = f'--resume {LATEST}' if os.path.exists(LATEST) else ''
print('Resume:', LATEST if RESUME_ARGS else 'none (new run)')
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    {RESUME_ARGS} --tensor-file alphatrain/data/born_again_A8.pt \
    --trunk p4m --num-blocks {NUM_BLOCKS} --group-channels {GROUP_CH} --p4m-expand {EXPAND} --seed {SEED} --amp {COMPILE} \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature 1.0 --blend-alpha 1.0 --policy-head pair2 --pair-dim 64 --legal-mask-loss \
    --save-every-steps {SAVE_STEPS} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/drive/MyDrive/alphatrain/{RUN}_ckpts 2>&1 | tee /content/{RUN}_train.log
# val loss is informational only - the gate is the local C++ eval (death rate over 2k games).

In [ ]:
import glob
for f in sorted(glob.glob(f'/content/drive/MyDrive/alphatrain/{RUN}_ckpts/*.pt')):
    print(f)

## Evaluation (local M5, C++ only)

Download epochs as they land (e.g. 7 / 14 / 20 / 27 / 33 / 40) into `alphatrain/data/`, then gate each one
(folded export, 2,000 games, cap 100k, logged to EVAL.md with the death rate):

```bash
python -m alphatrain.scripts.eval_log run --model alphatrain/data/<run>_ckpts_epoch_N.pt --desc "p4m 18b student of A8, epoch N"
```

Compare the death rate (deaths per 100k turns, 95% CI) with the teacher's; the teacher's 8-view-averaged
policy is the label source, so matching or beating the single-pass teacher is the goal.